In [1]:
import pandas as pd
import numpy as np
from passyunk.parser import PassyunkParser
import pyodbc
import sqlalchemy
import re
import Levenshtein
import glob

There is a new version of the Passyunk module available with updated data. 
Current: 2.34.0
Newest: 2.39.0
Run `pip install git+https://github.com/CityOfPhiladelphia/passyunk` to upgrade



# **Voter Registration Matching**

**Goal**: Leverage Voter Registration data from the Pennsylvania Department of State to identify owner-occupied properties in Philadelphia. The Department of Revenue does not formally maintain owner-occupancy status for Philadelphia parcels, yet this information is a key eligibility factor for several Real Estate Tax Assistance programs. Because property owners may use voter registration records to demonstrate owner-occupancy when applying for the [Homestead Exemption](https://www.phila.gov/services/payments-assistance-taxes/taxes/property-and-real-estate-taxes/get-real-estate-tax-relief/get-the-homestead-exemption/), this project focuses on standardizing, cleaning, and matching voter registration data to Philadelphia property ownership records to more accurately identify owner-occupied properties across the city.

Voter Registration data was purchased from the [Pennsylvania Department of State](https://www.pavoterservices.pa.gov/Pages/PurchasePAFULLVoterExport.aspx).

### Set year-month for final export

In [2]:
yearmn = '202507'

In [3]:
# date dataset purchased
load_date = '06/29/2025'

# **Import Datasets**

Before running, ensure that the voter registration data from the PA Department of State is downloaded and upzipped in the 02_data folder. The address and property ownership will be queried directly from REVALI,

## Import Voter Registration Data

In [4]:
col_names = ['id','1','last_name','first_name','middle_name','5','gender','date_birth','date_registered','status','date_status_change','party','address_num','address_suffix','street','unit',
    'company_name','city','state','zip_code','mailing_address','mailing_city','mailing_state','23','24','date_last_voted','26','27','28','29','30','31','32','33','34','35','36',
    '37','38','39','40','41','42','43','44','45','46','47','48','49','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66','67','68','69','70',
    '71','72','73','74','75','76','77','78','79','80','81','82','83','84','85','86','87','88','89','90','91','92','93','94','95','96','97','98','99','100','101','102','103',
    '104','105','106','107','108','109','110','111','112','113','114','115','116','117','118','119','120','121','122','123','124','125','126','127','128','129','130','131','132',
    '133','134','135','136','137','138','139','140','141','142','143','144','145','146','147','148','149','150','151','152','153']

In [5]:
voter = pd.read_csv("../02_data/Full Voter Export/PHILADELPHIA FVE 20250630.tsv", sep='\t', header=None, names=col_names)

C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\103773240.py:1: DtypeWarning: Columns (1,21,24,29,43,44,45,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,153) have mixed types. Specify dtype option on import or set low_memory=False.
  voter = pd.read_csv("../02_data/Full Voter Export/PHILADELPHIA FVE 20250630.tsv", sep='\t', header=None, names=col_names)


In [6]:
voter.shape

(1064215, 154)

## Connect to REVALI

In [7]:
# connection string 
ssms_connection_string = "Driver=SQL Server;Server=REVALIPDWH.city.phila.local;Database=NonCurrentFiles;TrustedConnection=yes;"
conn = pyodbc.connect(ssms_connection_string)

# build cursor
c = conn.cursor()

## Import Address Data

In [8]:
# set select statement
select_addr = """SELECT OPA#, flngCustomerKey, fstrStreet, fstrUnit, fstrZip, fcurMarketValue,
		ROW_NUMBER() OVER(PARTITION BY fstrStreet ORDER BY fcurMarketValue DESC) as rn
FROM [PRISM_tables].[dbo].[realEstateAssessment]
WHERE YEAR(fdtmFilingPeriod) = (SELECT YEAR(MAX(fdtmFilingPeriod)) FROM [PRISM_tables].[dbo].[realEstateAssessment])
AND fblnOPACease = 'false'
"""

In [9]:
try:
    c.execute(select_addr)
    address = c.fetchall()
    address = pd.DataFrame.from_records(address, columns=['OPA#', 'flngCustomerKey', 'fstrStreet', 'fstrUnit', 'fstrZip', 'fcurMarketValue', 'rn'])
    
except Exception as err:
    print(f"{type(err).__name__} was raised: {err}")


In [10]:
# keep only first record (with Market Value) for fstrStreet dupes
address = address.loc[address.rn ==1, ]
address = address.drop(['fcurMarketValue','rn'], axis = 1)

In [11]:
address.shape

(583803, 5)

## Import Property Owner Data

In [12]:
# set select statement
select_prop_own = """SELECT propertyId
		,fi64PropertyOwnerKey
		,fstrContactDataType
	  ,fstrListFormatName
FROM [PRISM_tables].[dbo].[propertyOwners]
WHERE fdtmTo = ''
AND fdtmInvalid = ''
AND fstrPropertyOwnerType <> 'BOBR'"""

In [13]:
try:
    c.execute(select_prop_own)
    prop_own = c.fetchall()
    prop_own = pd.DataFrame.from_records(prop_own, columns=['OPA', 'fi64PropertyOwnerKey','fstrContactDataType', 'fstrListFormatName'])
    
except Exception as err:
    print(f"{type(err).__name__} was raised: {err}")


In [14]:
prop_own.shape

(783695, 4)

# **Preprocess Datasets**

## Clean Voter Registration Data

In [15]:
# copy the df
voter_clean = voter.copy(deep=True)

# drop the unnecessary columns
voter_clean.drop(columns=['1', '5', '23', '24'], inplace=True)
voter_clean = voter_clean.loc[:,:'date_last_voted']

# drop inactive voters
voter_clean = voter_clean[voter_clean['status'] == 'A']

In [16]:
voter_clean.dtypes

id                    object
last_name             object
first_name            object
middle_name           object
gender                object
date_birth            object
date_registered       object
status                object
date_status_change    object
party                 object
address_num            int64
address_suffix        object
street                object
unit                  object
company_name          object
city                  object
state                 object
zip_code               int64
mailing_address       object
mailing_city          object
mailing_state         object
date_last_voted       object
dtype: object

In [17]:
voter_clean.isna().sum()

id                         0
last_name                 13
first_name                26
middle_name           322415
gender                 63819
date_birth                 1
date_registered            1
status                     0
date_status_change         0
party                      4
address_num                0
address_suffix        967229
street                     0
unit                  746799
company_name          936596
city                       0
state                      1
zip_code                   0
mailing_address       956756
mailing_city          970721
mailing_state         956475
date_last_voted       101581
dtype: int64

In [18]:
# drop records where name is missing
voter_clean.dropna(subset=['first_name', 'last_name'], inplace=True)

### Change data types

In [19]:
voter_clean = voter_clean.astype(object)

### Clean voter registration columns

In [20]:
# remove commas from data frame
voter_clean = voter_clean.replace(',', '', regex=True)
voter_clean.middle_name = voter_clean.middle_name.replace('', np.nan, regex=True) 


C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\848164597.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  voter_clean = voter_clean.replace(',', '', regex=True)


In [21]:
# create function to clean strings
def clean_str(x):
    return str(x).upper().strip() if pd.notnull(x) else np.nan

In [22]:
# apply clean_str function to standardized columns
for col in ['address_num','address_suffix','street','unit','city','state','zip_code','mailing_address','mailing_city','mailing_state','first_name','middle_name', 'last_name']:
    voter_clean[col] = voter_clean[col].apply(clean_str)

In [23]:
# remove special characters and spaces from unit
voter_clean.unit = voter_clean.unit.str.replace(r'[^a-zA-Z0-9]', '', regex=True)

# remove 'RD' as a unit designator
voter_clean.loc[voter_clean.unit == 'RD', 'unit'] = np.nan

# final clean of unit
voter_clean.unit = voter_clean.unit.apply(clean_str)

### Create full address field

In [24]:
# initialize full address
voter_clean['full_address'] = voter_clean['address_num'].astype(str) 

# add address suffix if exists
voter_clean.loc[voter_clean.address_suffix == '1/2','full_address'] += " 1/2"
voter_clean.loc[(voter_clean.address_suffix != '1/2') & (~voter_clean.address_suffix.isna()),'full_address'] += voter_clean['address_suffix'].astype(str)

# add street
voter_clean.full_address += ' ' + voter_clean['street'].astype(str) 

# add unit if exists
voter_clean.loc[~voter_clean.unit.isna(),'full_address'] += ' ' + voter_clean['unit'].astype(str)

# strip any extra spaces
voter_clean.full_address = voter_clean.full_address.str.replace('  ', ' ')


### Create full name fields

In [25]:
# separate last names for voters with two
voter_clean['last_name_1'] = voter_clean.last_name.apply(lambda x: x.split(' ')[0] if x.count(' ')==1 else np.nan)
voter_clean['last_name_2'] = voter_clean.last_name.apply(lambda x: x.split(' ')[-1] if x.count(' ')==1 else x)

In [26]:
# create full name from voter registration columns
voter_clean['full_name'] = voter_clean.last_name
voter_clean['full_name'] += ' ' + voter_clean.first_name
voter_clean.loc[~voter_clean.middle_name.isna(), 'full_name'] += ' ' + voter_clean.middle_name

# create full name variation (only middle initial)
voter_clean['middle_initial'] = voter_clean.middle_name.apply(lambda x: np.nan if pd.isna(x) else x[0])
voter_clean['full_name_2'] = voter_clean.last_name
voter_clean['full_name_2'] += ' ' + voter_clean.first_name
voter_clean.loc[~voter_clean.middle_initial.isna(), 'full_name_2'] += ' ' + voter_clean.middle_initial

# create full name variation (no middle name)
voter_clean['full_name_3'] = voter_clean.last_name  
voter_clean['full_name_3'] += ' ' + voter_clean.first_name

# create full name variation where double last names are separated
voter_clean['full_name_4'] = voter_clean.last_name_2 + ' ' + voter_clean.first_name
voter_clean.loc[~voter_clean.middle_name.isna(), 'full_name_4'] += ' ' + voter_clean.middle_name
voter_clean.loc[~voter_clean.last_name_1.isna(), 'full_name_4'] += ' ' + voter_clean.last_name_1

In [27]:
voter_clean.isna().sum()

id                         0
last_name                  0
first_name                 0
middle_name           322398
gender                 63816
date_birth                 1
date_registered            1
status                     0
date_status_change         0
party                      4
address_num                0
address_suffix        967190
street                     0
unit                  747229
company_name          936558
city                       0
state                      1
zip_code                   0
mailing_address       956717
mailing_city          970682
mailing_state         956436
date_last_voted       101573
full_address               0
last_name_1           932479
last_name_2                0
full_name                  0
middle_initial        322398
full_name_2                0
full_name_3                0
full_name_4                0
dtype: int64

### Standardize Addresses using [Passyunk Address Parser](https://github.com/CityOfPhiladelphia/passyunk)

In [28]:
## instantiate parser
p = PassyunkParser()

In [29]:
batch_size = 50000
results = []
batch_num = 1

for idx, row in enumerate(voter_clean.itertuples(), start=1):
    try:
        parsed = p.parse(row.full_address)
        results.append({
            "id": row.id,
            "std_address": parsed['components']['output_address'],
            "std_low_address": parsed['components']['address']['low_num'],
            "std_address_suffix": parsed['components']['address']['addr_suffix'],
            "std_street_name": parsed['components']['street']['name'],
            "std_street_dir": parsed['components']['street']['predir'],
            "std_street_suffix": parsed['components']['street']['suffix'],
            "std_unit": parsed['components']['address_unit']['unit_num'],
        })
    except Exception as err:
        results.append({
            "id": row.id,
            "std_address": np.nan,
            "std_low_address": np.nan,
            "std_address_suffix": np.nan,
            "std_street_name": np.nan,
            "std_street_dir": np.nan,
            "std_street_suffix": np.nan,
            "std_unit": np.nan,
        })

    # log progress every 10k rows
    if idx % 10000 == 0:
        print(f"Processed {idx:,} rows...")

    # dump batch every 50k rows
    if idx % batch_size == 0:
        out_path = f"../02_data/parsed_batch_{batch_num}.csv"
        pd.DataFrame(results).to_csv(out_path, index=False)
        print(f"Saved batch {batch_num} → {out_path}")
        results = []  # clear memory
        batch_num += 1

# save remainder if not empty
if results:
    out_path = f"../02_data/parsed_batch_{batch_num}.csv"
    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"Saved final batch {batch_num} → {out_path}")

Processed 10,000 rows...
Processed 20,000 rows...
Processed 30,000 rows...
Processed 40,000 rows...
Processed 50,000 rows...
Saved batch 1 → ../02_data/parsed_batch_1.csv
Processed 60,000 rows...
Processed 70,000 rows...
Processed 80,000 rows...
Processed 90,000 rows...
Processed 100,000 rows...
Saved batch 2 → ../02_data/parsed_batch_2.csv
Processed 110,000 rows...
Processed 120,000 rows...
Processed 130,000 rows...
Processed 140,000 rows...
Processed 150,000 rows...
Saved batch 3 → ../02_data/parsed_batch_3.csv
Processed 160,000 rows...
Processed 170,000 rows...
Processed 180,000 rows...
Processed 190,000 rows...
Processed 200,000 rows...
Saved batch 4 → ../02_data/parsed_batch_4.csv
Processed 210,000 rows...
Processed 220,000 rows...
Processed 230,000 rows...
Processed 240,000 rows...
Processed 250,000 rows...
Saved batch 5 → ../02_data/parsed_batch_5.csv
Processed 260,000 rows...
Processed 270,000 rows...
Processed 280,000 rows...
Processed 290,000 rows...
Processed 300,000 rows...

In [30]:
# get all CSVs that match the batch naming
batch_files = glob.glob("../02_data/parsed_batch_*.csv")

# read and concat them into one dataframe
parsed_all = pd.concat([pd.read_csv(f) for f in batch_files], ignore_index=True)

# merge parsed results back
voter_parsed = voter_clean.merge(parsed_all, on="id", how="left")

In [31]:
voter_clean.isna().sum()

id                         0
last_name                  0
first_name                 0
middle_name           322398
gender                 63816
date_birth                 1
date_registered            1
status                     0
date_status_change         0
party                      4
address_num                0
address_suffix        967190
street                     0
unit                  747229
company_name          936558
city                       0
state                      1
zip_code                   0
mailing_address       956717
mailing_city          970682
mailing_state         956436
date_last_voted       101573
full_address               0
last_name_1           932479
last_name_2                0
full_name                  0
middle_initial        322398
full_name_2                0
full_name_3                0
full_name_4                0
dtype: int64

### Clean Standardized Columns

In [32]:
# convert low/ address to strings
voter_parsed['std_low_address'] = voter_parsed['std_low_address'].astype(str)

# remove nans
voter_parsed['std_low_address'] = voter_parsed['std_low_address'].replace("nan", None) 

# remove the decimal values
voter_parsed['std_low_address'] = voter_parsed['std_low_address'].str[:-2]

In [33]:
# clean std_address field
voter_parsed.std_address = voter_parsed.std_address.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
voter_parsed.std_address = voter_parsed.std_address.str.replace(' SAINT ', ' ST ')
voter_parsed.std_address = voter_parsed.std_address.str.replace('OLD YORK ROAD RD', 'OLD YORK ROAD RD')

# clean std_street_name field
voter_parsed.std_street_name = voter_parsed.std_street_name.str.replace(' SAINT ', ' ST ')
voter_parsed.std_street_name = voter_parsed.std_street_name.str.replace('OLD YORK ROAD', 'OLD YORK')

# clean std_unit
voter_parsed.std_unit = voter_parsed.std_unit.str.replace(r'[^a-zA-Z0-9]', '', regex=True)

In [34]:
# apply clean_str function to standardized columns
for col in ['std_address','std_low_address','std_address_suffix','std_street_name','std_street_dir','std_street_suffix','std_unit','unit']:
    voter_parsed[col] = voter_parsed[col].apply(clean_str)

In [35]:
# fix Ayrdale Cresent addresses
voter_parsed.loc[(voter_parsed.std_street_name == 'AYRDALE CRESCENT') & (voter_parsed.std_unit.isna()) & (~voter_parsed['unit'].isna()), 'std_unit'] = voter_parsed.unit
voter_parsed.loc[voter_parsed.std_street_name == 'AYRDALE CRESCENT', 'std_street_suffix'] = 'CRES'
voter_parsed.loc[voter_parsed.std_street_name == 'AYRDALE CRESCENT', 'std_street_name'] = 'AYRDALE'

# fix 224-30 W Rittenhous Sq
voter_parsed.loc[voter_parsed.std_address == '226 W RITTENHOUSE SQ', 'std_low_address'] = '224'
voter_parsed.loc[voter_parsed.std_address == '226 W RITTENHOUSE SQ', 'std_address'] = '224 W RITTENHOUSE SQ'

In [36]:
# check the number of nans
voter_parsed.isna().sum()

id                         0
last_name                  0
first_name                 0
middle_name           322398
gender                 63816
date_birth                 1
date_registered            1
status                     0
date_status_change         0
party                      4
address_num                0
address_suffix        967190
street                     0
unit                  747229
company_name          936558
city                       0
state                      1
zip_code                   0
mailing_address       956717
mailing_city          970682
mailing_state         956436
date_last_voted       101573
full_address               0
last_name_1           932479
last_name_2                0
full_name                  0
middle_initial        322398
full_name_2                0
full_name_3                0
full_name_4                0
std_address                6
std_low_address            7
std_address_suffix    967712
std_street_name            6
std_street_dir

In [37]:
# remove records with missing low address
voter_parsed = voter_parsed.loc[~voter_parsed.std_low_address.isna()]

# remove records with missing street designation
voter_parsed = voter_parsed.loc[~voter_parsed.std_street_suffix.isna()]

In [38]:
voter_parsed.shape

(970670, 37)

# Preprocess PRISM Real Estate Data

In [39]:
address.dtypes

OPA#               object
flngCustomerKey    object
fstrStreet         object
fstrUnit           object
fstrZip            object
dtype: object

In [40]:
# split address into low, high, and suffix components where hyphen exists
address[['low_address', 'address_suffix', 'high_address_suffix']] = address['fstrStreet'].str.extract(
    r'^(\d+)([A-Za-z]?)-?(\d+)?\b'
)

# get address numbers before any letters, hypens, or spaces
address['low_address'] = address.fstrStreet.str.extract(r'^(.*?)(?=[A-Za-z- ])')

In [41]:
# address suffix
address.address_suffix = address.address_suffix.replace('', np.nan)

In [42]:
address.dtypes

OPA#                   object
flngCustomerKey        object
fstrStreet             object
fstrUnit               object
fstrZip                object
low_address            object
address_suffix         object
high_address_suffix    object
dtype: object

In [43]:
address.isna().sum()

OPA#                        0
flngCustomerKey             0
fstrStreet                  0
fstrUnit                    0
fstrZip                     0
low_address                 0
address_suffix         581427
high_address_suffix    557672
dtype: int64

In [44]:
# compute high address
def compute_high(row):
    if pd.notna(row['high_address_suffix']) and pd.notna(row['low_address']):
        low = str(row['low_address'])
        suffix = str(row['high_address_suffix'])
        return str(low[:-len(suffix)] + suffix)
    return np.nan

In [45]:
# create high address suffix
address['high_address'] = address.apply(compute_high, axis=1)

In [46]:
address.isna().sum()

OPA#                        0
flngCustomerKey             0
fstrStreet                  0
fstrUnit                    0
fstrZip                     0
low_address                 0
address_suffix         581427
high_address_suffix    557672
high_address           557672
dtype: int64

In [47]:
# clean fstrStreet field
address.fstrStreet = address.fstrStreet.str.replace(' MC ', ' MC')
address.fstrStreet = address.fstrStreet.str.replace(' O ', ' O')
address.fstrStreet = address.fstrStreet.str.replace('CHRIS COLUMBUS BLVD', 'CHRISTOPHER COLUMBUS BLVD')
address.fstrStreet = address.fstrStreet.str.replace('BEN FRANKLIN', 'BENJAMIN FRANKLIN')

# clean unit
address.fstrUnit = address.fstrUnit.str.replace(r'[^a-zA-Z0-9]', '', regex=True)
address.fstrUnit = address.fstrUnit.replace('', np.nan)

In [48]:
# apply clean_str function to address columns
for col in ['fstrStreet','fstrUnit','fstrZip','low_address','high_address_suffix','high_address']:
    address[col] = address[col].apply(clean_str)

In [49]:
address.isna().sum()

OPA#                        0
flngCustomerKey             0
fstrStreet                  0
fstrUnit               542439
fstrZip                     0
low_address                 0
address_suffix         581427
high_address_suffix    557672
high_address           557672
dtype: int64

## Preprocess Property Owner Data

In [50]:
prop_own.dtypes

OPA                     object
fi64PropertyOwnerKey    object
fstrContactDataType     object
fstrListFormatName      object
dtype: object

In [51]:
# apply clean_str function to property owner columns
for col in ['OPA', 'fstrContactDataType', 'fstrListFormatName']:
    prop_own[col] = prop_own[col].apply(clean_str)

In [52]:
# create indicator for middle name presence (more than one space in name)
prop_own['middle_name_bool'] = prop_own.fstrListFormatName.apply(lambda x: x.count(' ')> 1)

In [53]:
# create first and last name only field
prop_own['fstrFirstLast'] = prop_own.apply(lambda row: ' '.join(row.fstrListFormatName.split(' ')[:-1]) if row.middle_name_bool else row.fstrListFormatName, axis=1)

In [54]:
# drop the middle name indicator
prop_own.drop(columns=['middle_name_bool'], inplace=True)

# **Address Matching**

Match voter registration dataset to address dataset on address. 
- **Full Match**: full address matches exactly; 1:1 match
- **Partial Match**: parsed address components match; 1:1 match 
- **Fuzzy Match**: parsed address components match except for unit. 1:Many match; Perform additional match to property owner name to determine address match.

## Full Match

In [55]:
# convert all dtypes to string
voter_parsed = voter_parsed.astype(str)
address = address.astype(str)

# copy voter registrations to new unmatched df
unmatched = voter_parsed.copy(deep=True)

### Perform Full Match

In [56]:
matched = unmatched.merge(address, left_on=['std_address'], right_on=['fstrStreet'], suffixes=('_voter','_addr'))

In [57]:
matched.shape

(739926, 46)

### Remove duplicate matches

In [58]:
# count instances of each id
match_counts = matched.groupby('id').size().reset_index(name='count')

# valid ids only appear once
valid_ids = match_counts[match_counts['count'] == 1]['id']

# filter df to only include records matched once
matched = matched[matched['id'].isin(valid_ids)]

In [59]:
matched.shape

(739926, 46)

### Remove full matches from unmatched

In [60]:
unmatched = unmatched[~unmatched['id'].isin(valid_ids)]

In [61]:
unmatched.shape

(230744, 37)

# Partial Match

In [62]:
# set loop conditions
a = [i*10000 for i in range(unmatched.shape[0]//10000 + 1)]
a.append(unmatched.shape[0])

In [63]:
# create df to hold fuzzy matches
fuzzy_matched = matched[0:0].copy(deep=True)

In [64]:
for i in range(len(a)-1):
    # set unmatched batch
    print(f"Processing unmatched batch {a[i]} to {a[i+1]}.")
    unmatched_batch = unmatched[a[i]:a[i+1]]

    ## create low and high address joins
    low_addr_match = unmatched_batch.merge(address, left_on=['std_low_address'], right_on=['low_address'], suffixes=('_voter','_addr')) 
    high_addr_match = unmatched_batch.merge(address, left_on=['std_low_address'], right_on=['high_address'], suffixes=('_voter','_addr')) 


    ## create match indicators
    # address suffix indicator
    low_addr_match['address_suffix_match'] = low_addr_match.address_suffix_addr == low_addr_match.std_address_suffix
    high_addr_match['address_suffix_match'] = high_addr_match.address_suffix_addr == high_addr_match.std_address_suffix

    # street direction match
    low_addr_match['street_dir_match'] = low_addr_match.apply(
        lambda row: row['std_street_dir'] == 'nan' or
                    bool(re.search(rf"\b{re.escape(str(row['std_street_dir']))}\b", str(row['fstrStreet']))),
        axis=1
    )
    high_addr_match['street_dir_match'] = high_addr_match.apply(
        lambda row: row['std_street_dir'] == 'nan' or
                    bool(re.search(rf"\b{re.escape(str(row['std_street_dir']))}\b", str(row['fstrStreet']))),
        axis=1
    )

    # street name match
    low_addr_match['street_name_match'] = low_addr_match.apply(
        lambda row: bool(re.search(rf"\b{re.escape(str(row['std_street_name']))}\b", str(row['fstrStreet']))),
        axis=1
    )
    
    high_addr_match['street_name_match'] = high_addr_match.apply(
        lambda row: bool(re.search(rf"\b{re.escape(str(row['std_street_name']))}\b", str(row['fstrStreet']))),
        axis=1
    )

    # street suffix match
    low_addr_match['street_suffix_match'] = low_addr_match.apply(
        lambda row: row['std_street_suffix'] == 'nan' or
                    bool(re.search(rf"\b{re.escape(str(row['std_street_suffix']))}\b", str(row['fstrStreet']))),
        axis=1
    )
    high_addr_match['street_suffix_match'] = high_addr_match.apply(
        lambda row: row['std_street_suffix'] == 'nan' or
                    bool(re.search(rf"\b{re.escape(str(row['std_street_suffix']))}\b", str(row['fstrStreet']))),
        axis=1
    )

    # unit indicator
    low_addr_match['unit_match'] = low_addr_match.std_unit == low_addr_match.fstrUnit
    low_addr_match['unit_match_2'] = low_addr_match.unit == low_addr_match.fstrUnit
    high_addr_match['unit_match'] = high_addr_match.std_unit == high_addr_match.fstrUnit
    high_addr_match['unit_match_2'] = high_addr_match.unit == high_addr_match.fstrUnit

    # 1/2 address match
    low_addr_match['half_match'] = (
        low_addr_match['std_address'].str.contains('1/2', na=False) ==
        low_addr_match['fstrStreet'].str.contains('1/2', na=False)
    )
    high_addr_match['half_match'] = (
        high_addr_match['std_address'].str.contains('1/2', na=False) ==
        high_addr_match['fstrStreet'].str.contains('1/2', na=False)
    )

    # no address suffix match
    low_addr_match['no_address_suffix_match'] = low_addr_match.std_address_suffix == 'nan'
    high_addr_match['no_address_suffix_match'] = high_addr_match.std_address_suffix == 'nan'


    ## create match cases
    # match 1: all standardized components match
    low_addr_match['match_case_1'] = low_addr_match[[
        'address_suffix_match',
        'street_dir_match',
        'street_name_match',
        'street_suffix_match',
        'unit_match',
        'half_match'
    ]].all(axis=1)

    # match 2: all standardized components match except low address
    # high address match, excludes case with address suffix
    high_addr_match['match_case_2'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['street_suffix_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # match 3: all standardized components match except street suffix
    low_addr_match['match_case_3'] = (
        low_addr_match['address_suffix_match'] &
        low_addr_match['street_dir_match'] &
        low_addr_match['street_name_match'] &
        low_addr_match['unit_match'] &
        low_addr_match['half_match']
    )
    # match 4: all standardized components match except low address and street suffix
    # high address match, excludes uses with address suffix
    high_addr_match['match_case_4'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # match 5: all standardized components match except std_unit
    # voter registration unit match
    low_addr_match['match_case_5'] = (
        low_addr_match['address_suffix_match'] &
        low_addr_match['street_dir_match'] &
        low_addr_match['street_name_match'] &
        low_addr_match['street_suffix_match'] &
        low_addr_match['unit_match_2'] &
        low_addr_match['half_match']
    )
    # match 6: all standardized components match except low address and std_unit
    # high address match, excludes case with address suffix
    # voter registration unit match
    high_addr_match['match_case_6'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['street_suffix_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # match 7: all standardized components match except std_unit and street suffix
    # voter registration unit match
    low_addr_match['match_case_7'] = (
        low_addr_match['address_suffix_match'] &
        low_addr_match['street_dir_match'] &
        low_addr_match['street_name_match'] &
        low_addr_match['unit_match_2'] &
        low_addr_match['half_match']
    )
    # match 8: all standardized components match except low address, std_unit, and street suffix
    # high address match, excludes case with address suffix
    # voter registration unit match
    high_addr_match['match_case_8'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # match 9: all standardized components match except street direction
    low_addr_match['match_case_9'] = low_addr_match[[
        'address_suffix_match',
        'street_name_match',
        'street_suffix_match',
        'unit_match',
        'half_match'
    ]].all(axis=1)

    # match 10: all standardized components match except low address and street direction
    # high address match, excludes case with address suffix
    high_addr_match['match_case_10'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['street_suffix_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # match 11: all standardized components match except unit
    # no units for address
    low_addr_match['match_case_11'] = low_addr_match[[
        'address_suffix_match',
        'street_dir_match',
        'street_name_match',
        'street_suffix_match',
        'half_match'
    ]].all(axis=1)

    # match 12: all standardized components match except low address and unit
    # high address match, excludes case with address suffix
    # no units for address
    high_addr_match['match_case_12'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['street_suffix_match'] &
        high_addr_match['unit_match'] &
        high_addr_match['half_match']
    )
    # fuzzy 1: all standardized components match except unit
    low_addr_match['fuzzy_match_1'] = low_addr_match[[
        'address_suffix_match',
        'street_dir_match',
        'street_name_match',
        'street_suffix_match',
        'half_match'
    ]].all(axis=1)
    # fuzzy 2: all standardized components match except low address and unit
    # high address match, excludes case with address suffix
    high_addr_match['fuzzy_match_2'] = (
        high_addr_match['no_address_suffix_match'] &
        high_addr_match['street_dir_match'] &
        high_addr_match['street_name_match'] &
        high_addr_match['street_suffix_match'] &
        high_addr_match['half_match']
    )


    ## perform low address match
    for m in ['match_case_1', 'match_case_3', 'match_case_5', 'match_case_7','match_case_9','match_case_11','fuzzy_match_1']:
        # get matching records by case
        temp_match = low_addr_match.loc[low_addr_match[m],:'high_address'].copy(deep=True)

        if m != 'fuzzy_match_1':
            # count instances of each id
            match_counts = temp_match.groupby('id').size().reset_index(name='count')

            # valid ids only appear once
            valid_ids = match_counts[match_counts['count'] == 1]['id']

            # filter df to only include records matched once
            temp_match = temp_match[temp_match['id'].isin(valid_ids)]

            print(f"Number of records matched by {m}: {temp_match.shape[0]}")
            
            # add to matched df
            matched = pd.concat([matched, temp_match], ignore_index=True) 

            # remove matched records from low_addr_match
            low_addr_match = low_addr_match[~low_addr_match['id'].isin(valid_ids)]
            high_addr_match = high_addr_match[~high_addr_match['id'].isin(valid_ids)]
        else:
            print(f"Number of records matched by {m}: {temp_match.shape[0]}")
            # add fuzzy results
            fuzzy_matched = temp_match.copy(deep=True)



    ## perform high address match
    for m in ['match_case_2', 'match_case_4', 'match_case_6', 'match_case_8', 'match_case_10', 'match_case_12', 'fuzzy_match_2']:
        # get matching records by case
        temp_match = high_addr_match.loc[high_addr_match[m],:'high_address'].copy(deep=True)

        if m != 'fuzzy_match_2':
            # count instances of each id
            match_counts = temp_match.groupby('id').size().reset_index(name='count')

            # valid ids only appear once
            valid_ids = match_counts[match_counts['count'] == 1]['id']

            # filter df to only include records matched once
            temp_match = temp_match[temp_match['id'].isin(valid_ids)]

            print(f"Number of records matched by {m}: {temp_match.shape[0]}")

            # add to matched df
            matched = pd.concat([matched, temp_match], ignore_index=True) 

            # remove matched records from low_addr_match
            low_addr_match = low_addr_match[~low_addr_match['id'].isin(valid_ids)]
            high_addr_match = high_addr_match[~high_addr_match['id'].isin(valid_ids)]
        else:
            print(f"Number of records matched by {m}: {temp_match.shape[0]}")
            # add fuzzy results
            fuzzy_matched = pd.concat([fuzzy_matched, temp_match], ignore_index=True)

            # export fuzzy matches
            fuzzy_matched.to_csv(f"../02_data/Fuzzy_Matched_Voter_Registration_{a[i+1]}.csv", index=False) 

    # get unique list of the rest of the ids
    rest_ids = list(pd.concat([low_addr_match.id, high_addr_match.id]).unique())

    # get not matched records
    not_matched = unmatched_batch[unmatched_batch['id'].isin(rest_ids)]

    print(f'Number of matched records after address matching: {unmatched_batch.shape[0] - not_matched.shape[0]}')
    print(f'Number of unmatched records after address matching: {not_matched.shape[0]}')

Processing unmatched batch 0 to 10000.
Number of records matched by match_case_1: 3552
Number of records matched by match_case_3: 143
Number of records matched by match_case_5: 2289
Number of records matched by match_case_7: 0
Number of records matched by match_case_9: 366
Number of records matched by match_case_11: 380
Number of records matched by fuzzy_match_1: 83787
Number of records matched by match_case_2: 407
Number of records matched by match_case_4: 1
Number of records matched by match_case_6: 0
Number of records matched by match_case_8: 0
Number of records matched by match_case_10: 7
Number of records matched by match_case_12: 0
Number of records matched by fuzzy_match_2: 1084
Number of matched records after address matching: 7167
Number of unmatched records after address matching: 2833
Processing unmatched batch 10000 to 20000.
Number of records matched by match_case_1: 4383
Number of records matched by match_case_3: 180
Number of records matched by match_case_5: 727
Number o

In [65]:
matched.shape

(902844, 46)

In [66]:
voter_parsed.shape

(970670, 37)

In [67]:
(901989 + 2830)/970647 

0.9321813182341263

In [68]:
# get all CSVs that match the batch naming
fuzzy_files = glob.glob("../02_data/Fuzzy_Matched_Voter_Registration_*.csv")

# read and concat them into one dataframe
fuzzy = pd.concat([pd.read_csv(f) for f in fuzzy_files], ignore_index=True)


C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\3837829728.py:5: DtypeWarning: Columns (18,20,23,36) have mixed types. Specify dtype option on import or set low_memory=False.
  fuzzy = pd.concat([pd.read_csv(f) for f in fuzzy_files], ignore_index=True)
C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\3837829728.py:5: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  fuzzy = pd.concat([pd.read_csv(f) for f in fuzzy_files], ignore_index=True)
C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\3837829728.py:5: DtypeWarning: Columns (36,43) have mixed types. Specify dtype option on import or set low_memory=False.
  fuzzy = pd.concat([pd.read_csv(f) for f in fuzzy_files], ignore_index=True)
C:\Users\julia.f.flanagan\AppData\Local\Temp\ipykernel_7556\3837829728.py:5: DtypeWarning: Columns (43) have mixed types. Specify dtype option on import or set low_memory=False.
  fuzzy = pd.concat([pd.read_csv(f) for f 

In [69]:
fuzzy.shape

(805779, 46)

In [70]:
fuzzy = fuzzy.astype(object)

In [71]:
fuzzy['OPA#']= fuzzy['OPA#'].apply(clean_str)

# **Property Owner Name Matching**

In [72]:
# voter records matched to addresses x property owners for those addresses
voter_owner = pd.merge(matched, prop_own, left_on='OPA#', right_on='OPA', how='left', suffixes=('_voter','_prop'))

# voter records fuzzy matched to addresses x property owners for those addresses
fuzzy_voter_owner = pd.merge(fuzzy, prop_own, left_on='OPA#', right_on='OPA', how='left', suffixes=('_voter','_prop'))

## Calculate Levenshtein Similarity Scores

In [73]:
# Calculate levenshtein similarity ratio for matched results

voter_owner['similarity_ratio'] = voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name']),
    axis=1)

voter_owner['similarity_ratio_2'] = voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_2']),
    axis=1)

voter_owner['similarity_ratio_3'] = voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_3']),
    axis=1)

# two last names
voter_owner['similarity_ratio_4'] = voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_4']),
    axis=1)

# middle name missing in PRISM // 0.1 penalty for removing part of PRISM List Format Name
voter_owner['similarity_ratio_5'] = voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrFirstLast'], row['full_name_3']) - .1,
    axis=1)

voter_owner['max_similarity'] = voter_owner[['similarity_ratio', 'similarity_ratio_2','similarity_ratio_3','similarity_ratio_4','similarity_ratio_5']].max(axis=1)


In [74]:
# Calculate levenshtein similarity ratio for fuzzy matched results

fuzzy_voter_owner['similarity_ratio'] = fuzzy_voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name']),
    axis=1)

fuzzy_voter_owner['similarity_ratio_2'] = fuzzy_voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_2']),
    axis=1)

fuzzy_voter_owner['similarity_ratio_3'] = fuzzy_voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_3']),
    axis=1)

# two last names
fuzzy_voter_owner['similarity_ratio_4'] = fuzzy_voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrListFormatName'], row['full_name_4']),
    axis=1)

# middle name missing in PRISM // 0.1 penalty for removing part of PRISM List Format Name
fuzzy_voter_owner['similarity_ratio_5'] = fuzzy_voter_owner.apply(
    lambda row: Levenshtein.ratio(row['fstrFirstLast'], row['full_name_3']) - .1,
    axis=1)

fuzzy_voter_owner['max_similarity'] = fuzzy_voter_owner[['similarity_ratio', 'similarity_ratio_2','similarity_ratio_3','similarity_ratio_4','similarity_ratio_5']].max(axis=1)

## Select Best Property Owner Match for Each Address

In [75]:
voter_owner['best_match'] = (
    voter_owner.sort_values(['id', 'max_similarity'], ascending=[True, False])
      .groupby('id')
      .cumcount() + 1
)

best_match = voter_owner.loc[voter_owner.best_match == 1,]

In [76]:
fuzzy_voter_owner['best_match'] = (
    fuzzy_voter_owner.sort_values(['id', 'max_similarity'], ascending=[True, False])
      .groupby('id')
      .cumcount() + 1
)

best_fuzzy_match = fuzzy_voter_owner.loc[fuzzy_voter_owner.best_match == 1,]

In [77]:
best_match.shape

(902844, 58)

In [78]:
best_fuzzy_match.shape

(11301, 58)

In [79]:
best_match.loc[best_match.max_similarity >= .8,].shape 

(259181, 58)

In [80]:
final_fuzzy_match = best_fuzzy_match.loc[best_fuzzy_match.max_similarity >= .8,] 

In [81]:
final_fuzzy_match.shape # 2830

(2871, 58)

# **Finalize and Export**

### Combined best matches from full, partial, and fuzzy match dfs

In [82]:
final = pd.concat([best_match, final_fuzzy_match])

In [83]:
final.shape

(905715, 58)

In [84]:
final.dtypes

id                       object
last_name                object
first_name               object
middle_name              object
gender                   object
date_birth               object
date_registered          object
status                   object
date_status_change       object
party                    object
address_num              object
address_suffix_voter     object
street                   object
unit                     object
company_name             object
city                     object
state                    object
zip_code                 object
mailing_address          object
mailing_city             object
mailing_state            object
date_last_voted          object
full_address             object
last_name_1              object
last_name_2              object
full_name                object
middle_initial           object
full_name_2              object
full_name_3              object
full_name_4              object
std_address              object
std_low_

### Remove Property Owner Key for name matches that don't meet 0.8 threshold

In [85]:
final.loc[final.max_similarity < .8, "fi64PropertyOwnerKey"] = 0

In [86]:
# drop unnecessary columns
final.drop(['full_address', 'std_address',
       'std_low_address', 'std_address_suffix', 'std_street_name',
       'std_street_dir', 'std_street_suffix', 'std_unit',
       'flngCustomerKey', 'fstrStreet', 'fstrUnit', 'fstrZip', 'low_address',
       'address_suffix_addr', 'high_address_suffix', 'high_address',
       'full_name', 'middle_initial', 'full_name_2', 'full_name_3', 'OPA',
       'fstrContactDataType', 'fstrListFormatName', 'similarity_ratio',
       'similarity_ratio_2', 'similarity_ratio_3', 'max_similarity',
       'best_match','party','gender'], 
       axis=1, inplace=True
       )

In [87]:
# rename columns
final = final.rename(columns={'id':'voter_id',
                              'status':'voter_status',
                              'address_suffix_voter':'address_suffix'})

In [88]:
# create identifier for whether an owner match was found
final['fblnOwnerIdentied'] = final.fi64PropertyOwnerKey.apply(lambda x: 1 if x != 0 else 0)

In [89]:
# create load date
final['load_date'] = load_date

In [90]:
# reorder columns
final = final.reindex(columns = ['voter_id', 'OPA#', 'fi64PropertyOwnerKey', 'fblnOwnerIdentied', 'last_name', 'first_name', 'middle_name',
       'date_birth', 'date_registered', 'voter_status', 'date_status_change',
       'address_num', 'address_suffix', 'street', 'unit',
       'company_name', 'city', 'state', 'zip_code', 'mailing_address',
       'mailing_city', 'mailing_state', 'date_last_voted',  'load_date'])

In [91]:
final = final.replace('nan', None)

In [92]:
sample = final.iloc[:50]

### Export final dataset

In [93]:
# final table for SQL Server
final.to_csv(f'../02_data/Voter_Registration_Philadelphia_{yearmn}.csv', index=False)

# final table for PRISM
final.to_csv(f'../02_data/Voter_Registration_Philadelphia_PRISM_{yearmn}.csv', index=False, header=False)

# sample file for data warehouse
sample.to_csv(f'../02_data/Voter_Registration_Philadelphia_PRISM_{yearmn}_SAMPLE.csv', index=False, header=False)

In [94]:
#unmatched.loc[(~unmatched.id.isin(final_fuzzy_match['id'])) & (~unmatched.id.isin(final['voter_id'])), ].to_csv('../02_data/check_unmatched.csv', index=False)